In [1]:
#DNA BERT :

import torch
import torch.nn as nn
import numpy as np
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, matthews_corrcoef, roc_auc_score
from collections import Counter
import random
import re
from tqdm import tqdm

# --- Reproducibility ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

# --- Data Utilities ---
def clean_sequence(seq):
    seq = seq.upper()
    return re.sub('[^ATCG]', '', seq)

def load_fasta_clean(file_path):
    with open(file_path) as f:
        lines = f.readlines()
    return [clean_sequence(lines[i+1].strip()) for i in range(0, len(lines), 2)]

def reverse_complement(seq):
    complement = str.maketrans("ATCG", "TAGC")
    return seq.translate(complement)[::-1]

def kmer_tokenize(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

# --- Shared Dataset Class ---
class SimpleDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=512):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        encoding = self.tokenizer(self.sequences[idx], padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.float),
        }

# --- DNA BERT Model Class (Only DNA BERT) ---
class DNABERT(nn.Module):
    def __init__(self, dropout_rate=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained("zhihan1996/DNA_bert_6")
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(self.bert.config.hidden_size, 1)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(outputs.pooler_output)

# --- Evaluation Utility ---
def evaluate(model, loader, device):
    model.eval()
    preds, trues = [], []
    tp, tn, fp, fn = 0, 0, 0, 0  # Initialize confusion matrix counts
    all_probs = []  # List to store all probability values

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)
            logits = model(input_ids, attention_mask).squeeze()
            probs = torch.sigmoid(logits)
            preds.extend((probs > 0.5).int().cpu().tolist())
            trues.extend(label.int().cpu().tolist())
            all_probs.extend(probs.cpu().tolist())  # Collect all probs

            # Update confusion matrix
            for true, pred in zip(label.int().cpu().tolist(), (probs > 0.5).int().cpu().tolist()):
                if true == 1 and pred == 1:
                    tp += 1  # True Positive
                elif true == 0 and pred == 0:
                    tn += 1  # True Negative
                elif true == 0 and pred == 1:
                    fp += 1  # False Positive
                elif true == 1 and pred == 0:
                    fn += 1  # False Negative

    # Accuracy
    accuracy = accuracy_score(trues, preds)
    # F1 Score
    f1 = f1_score(trues, preds)
    # Specificity
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    # Sensitivity
    sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0
    # MCC
    mcc = matthews_corrcoef(trues, preds)

    # AUC - Ensure probs are in numpy format and length matches
    all_probs_np = np.array(all_probs).flatten()
    trues_np = np.array(trues)
    auc = roc_auc_score(trues_np, all_probs_np)

    print(f"Accuracy: {accuracy:.4f} | F1 Score: {f1:.4f} | Sensitivity (SN): {sensitivity:.4f} | Specificity (SP): {specificity:.4f} | MCC: {mcc:.4f} | AUC: {auc:.4f}")
    print("Confusion Matrix:\n", [[tp, fp], [fn, tn]])

    return accuracy, f1, sensitivity, specificity, mcc, auc

# === STAGE 1: ENHANCER VS NON-ENHANCER ===
def prepare_stage1():
    enhancer_train = load_fasta_clean("/kaggle/input/tahapu/enhancer.cv.txt")
    non_enhancer_train = load_fasta_clean("/kaggle/input/tahapu/non.cv.txt")
    
    enhancer_test = load_fasta_clean("/kaggle/input/tahapu/enhancer.ind.txt")
    non_enhancer_test = load_fasta_clean("/kaggle/input/tahapu/non.ind.txt")

    min_len_train = min(len(enhancer_train), len(non_enhancer_train))
    enhancer_train = enhancer_train[:min_len_train]
    non_enhancer_train = non_enhancer_train[:min_len_train]

    min_len_test = min(len(enhancer_test), len(non_enhancer_test))
    enhancer_test = enhancer_test[:min_len_test]
    non_enhancer_test = non_enhancer_test[:min_len_test]

    X_train, y_train = [], []
    X_test, y_test = [], []

    for seq in enhancer_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(1)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(1)
    for seq in non_enhancer_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(0)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(0)

    for seq in enhancer_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(1)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(1)
    for seq in non_enhancer_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(0)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(0)

    return X_train, y_train, X_test, y_test


# === STAGE 2: STRONG VS WEAK ENHANCERS ===
def prepare_stage2():
    strong_train = load_fasta_clean("/kaggle/input/tahapu/strong_742.txt")
    weak_train = load_fasta_clean("/kaggle/input/tahapu/weak_742.txt")

    strong_test = load_fasta_clean("/kaggle/input/tahapu/strong_100.txt")
    weak_test = load_fasta_clean("/kaggle/input/tahapu/weak_100.txt")

    max_len_train = max(len(strong_train), len(weak_train))
    strong_train = (strong_train * ((max_len_train // len(strong_train)) + 1))[:max_len_train]
    weak_train = (weak_train * ((max_len_train // len(weak_train)) + 1))[:max_len_train]

    max_len_test = max(len(strong_test), len(weak_test))
    strong_test = (strong_test * ((max_len_test // len(strong_test)) + 1))[:max_len_test]
    weak_test = (weak_test * ((max_len_test // len(weak_test)) + 1))[:max_len_test]

    X_train, y_train = [], []
    X_test, y_test = [], []

    for seq in strong_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(1)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(1)
    for seq in weak_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(0)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(0)

    for seq in strong_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(1)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(1)
    for seq in weak_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(0)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(0)

    return X_train, y_train, X_test, y_test

def train_stage1():
    tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNA_bert_6", do_lower_case=False, trust_remote_code=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Prepare data
    X_train, y_train, X_test, y_test = prepare_stage1()
    print("Train Label Distribution:", Counter(y_train))
    print("Test Label Distribution:", Counter(y_test))

    train_ds = SimpleDataset(X_train, y_train, tokenizer)
    test_ds = SimpleDataset(X_test, y_test, tokenizer)
    batch_size = 8  # Starting batch size
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    model = DNABERT(dropout_rate=0.3).to(device)  # Using only DNA BERT
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    # Learning rate scheduler (Cosine Annealing)
    scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

    best_f1 = 0
    for epoch in range(10):  # 10 epochs
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Stage 1 - Epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask).squeeze()
            loss = loss_fn(logits, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")
        evaluate(model, test_loader, device)

        # Update learning rate based on scheduler
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                label = batch["label"].to(device)
                logits = model(input_ids, attention_mask).squeeze()
                preds.extend((torch.sigmoid(logits) > 0.5).int().cpu().tolist())
                trues.extend(label.int().cpu().tolist())
        f1 = f1_score(trues, preds)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "enhancer_stage1.pth")
            print("✅ Stage 1 model saved.")

def train_stage2():
    tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNA_bert_6", do_lower_case=False, trust_remote_code=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Prepare data
    X_train, y_train, X_test, y_test = prepare_stage2()
    print("Train Label Distribution:", Counter(y_train))
    print("Test Label Distribution:", Counter(y_test))

    train_ds = SimpleDataset(X_train, y_train, tokenizer)
    test_ds = SimpleDataset(X_test, y_test, tokenizer)
    batch_size = 8  # Starting batch size
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    model = DNABERT(dropout_rate=0.3).to(device)  # Using only DNA BERT
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    # Learning rate scheduler (Cosine Annealing)
    scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

    best_f1 = 0
    for epoch in range(10):  # 10 epochs
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Stage 2 - Epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask).squeeze()
            loss = loss_fn(logits, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")
        evaluate(model, test_loader, device)

        # Update learning rate based on scheduler
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                label = batch["label"].to(device)
                logits = model(input_ids, attention_mask).squeeze()
                preds.extend((torch.sigmoid(logits) > 0.5).int().cpu().tolist())
                trues.extend(label.int().cpu().tolist())
        f1 = f1_score(trues, preds)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "enhancer_stage2.pth")
            print("✅ Stage 2 model saved.")

# === MAIN ===
if __name__ == "__main__":
    train_stage1()
    train_stage2()


tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

configuration_bert.py:   0%|          | 0.00/807 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/zhihan1996/DNA_bert_6:
- configuration_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


vocab.txt:   0%|          | 0.00/28.7k [00:00<?, ?B/s]

...

Epoch 8 Loss: 0.4408
Accuracy: 0.6310 | F1 Score: 0.5125 | Sensitivity (SN): 0.3880 | Specificity (SP): 0.8740 | MCC: 0.2998 | AUC: 0.6937
Confusion Matrix:
 [[194, 63], [306, 437]]


Stage 2 - Epoch 9: 100%|██████████| 928/928 [11:47<00:00,  1.31it/s]


Epoch 9 Loss: 0.4197
Accuracy: 0.5700 | F1 Score: 0.4905 | Sensitivity (SN): 0.4140 | Specificity (SP): 0.7260 | MCC: 0.1474 | AUC: 0.6947
Confusion Matrix:
 [[207, 137], [293, 363]]


Stage 2 - Epoch 10: 100%|██████████| 928/928 [11:48<00:00,  1.31it/s]


Epoch 10 Loss: 0.3975
Accuracy: 0.6180 | F1 Score: 0.5319 | Sensitivity (SN): 0.4340 | Specificity (SP): 0.8020 | MCC: 0.2538 | AUC: 0.6945
Confusion Matrix:
 [[217, 99], [283, 401]]


**DNABERT + BiLSTM :**

In [1]:
import torch
import torch.nn as nn
import numpy as np
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, matthews_corrcoef, roc_auc_score
from collections import Counter
import random
import re
from tqdm import tqdm

# --- Reproducibility ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

# --- Data Utilities ---
def clean_sequence(seq):
    seq = seq.upper()
    return re.sub('[^ATCG]', '', seq)

def load_fasta_clean(file_path):
    with open(file_path) as f:
        lines = f.readlines()
    return [clean_sequence(lines[i+1].strip()) for i in range(0, len(lines), 2)]

def reverse_complement(seq):
    complement = str.maketrans("ATCG", "TAGC")
    return seq.translate(complement)[::-1]

def kmer_tokenize(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

# --- Shared Dataset Class ---
class SimpleDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=512):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        encoding = self.tokenizer(self.sequences[idx], padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.float),
        }

# --- DNA BERT + BiLSTM Model ---
class DNABERT_BiLSTM(nn.Module):
    def __init__(self, dropout_rate=0.3, lstm_hidden_size=128, lstm_num_layers=2):
        super().__init__()
        # Load pre-trained DNA BERT model
        self.bert = AutoModel.from_pretrained("zhihan1996/DNA_bert_6")
        
        # BiLSTM Layer
        self.lstm = nn.LSTM(input_size=self.bert.config.hidden_size, 
                            hidden_size=lstm_hidden_size, 
                            num_layers=lstm_num_layers, 
                            bidirectional=True, 
                            batch_first=True)
        
        # Fully connected layer
        self.fc1 = nn.Linear(lstm_hidden_size * 2, 1)  # *2 for bidirectional LSTM
        
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout_rate)
        
    def forward(self, input_ids, attention_mask):
        # Pass through BERT
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Get the sequence output (not pooler output)
        x = outputs.last_hidden_state
        
        # Pass through BiLSTM
        x, _ = self.lstm(x)
        
        # Use the representation of the last hidden state
        x = x[:, -1, :]  # Get last hidden state
        
        # Pass through dropout and fully connected layer
        x = self.dropout(x)
        x = self.fc1(x)
        
        return x

# --- Evaluation Utility ---
def evaluate(model, loader, device):
    model.eval()
    preds, trues = [], []
    tp, tn, fp, fn = 0, 0, 0, 0  # Initialize confusion matrix counts
    all_probs = []  # List to store all probability values

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)
            logits = model(input_ids, attention_mask).squeeze()
            probs = torch.sigmoid(logits)
            preds.extend((probs > 0.5).int().cpu().tolist())
            trues.extend(label.int().cpu().tolist())
            all_probs.extend(probs.cpu().tolist())  # Collect all probs

            # Update confusion matrix
            for true, pred in zip(label.int().cpu().tolist(), (probs > 0.5).int().cpu().tolist()):
                if true == 1 and pred == 1:
                    tp += 1  # True Positive
                elif true == 0 and pred == 0:
                    tn += 1  # True Negative
                elif true == 0 and pred == 1:
                    fp += 1  # False Positive
                elif true == 1 and pred == 0:
                    fn += 1  # False Negative

    # Accuracy
    accuracy = accuracy_score(trues, preds)
    # F1 Score
    f1 = f1_score(trues, preds)
    # Specificity
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    # Sensitivity
    sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0
    # MCC
    mcc = matthews_corrcoef(trues, preds)

    # AUC - Ensure probs are in numpy format and length matches
    all_probs_np = np.array(all_probs).flatten()
    trues_np = np.array(trues)
    auc = roc_auc_score(trues_np, all_probs_np)

    print(f"Accuracy: {accuracy:.4f} | F1 Score: {f1:.4f} | Sensitivity (SN): {sensitivity:.4f} | Specificity (SP): {specificity:.4f} | MCC: {mcc:.4f} | AUC: {auc:.4f}")
    print("Confusion Matrix:\n", [[tp, fp], [fn, tn]])

    return accuracy, f1, sensitivity, specificity, mcc, auc

# === STAGE 1: ENHANCER VS NON-ENHANCER ===
def prepare_stage1():
    enhancer_train = load_fasta_clean("/kaggle/input/tahapu/enhancer.cv.txt")
    non_enhancer_train = load_fasta_clean("/kaggle/input/tahapu/non.cv.txt")
    
    enhancer_test = load_fasta_clean("/kaggle/input/tahapu/enhancer.ind.txt")
    non_enhancer_test = load_fasta_clean("/kaggle/input/tahapu/non.ind.txt")

    min_len_train = min(len(enhancer_train), len(non_enhancer_train))
    enhancer_train = enhancer_train[:min_len_train]
    non_enhancer_train = non_enhancer_train[:min_len_train]

    min_len_test = min(len(enhancer_test), len(non_enhancer_test))
    enhancer_test = enhancer_test[:min_len_test]
    non_enhancer_test = non_enhancer_test[:min_len_test]

    X_train, y_train = [], []
    X_test, y_test = [], []

    for seq in enhancer_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(1)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(1)
    for seq in non_enhancer_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(0)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(0)

    for seq in enhancer_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(1)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(1)
    for seq in non_enhancer_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(0)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(0)

    return X_train, y_train, X_test, y_test


# === STAGE 2: STRONG VS WEAK ENHANCERS ===
def prepare_stage2():
    strong_train = load_fasta_clean("/kaggle/input/tahapu/strong_742.txt")
    weak_train = load_fasta_clean("/kaggle/input/tahapu/weak_742.txt")

    strong_test = load_fasta_clean("/kaggle/input/tahapu/strong_100.txt")
    weak_test = load_fasta_clean("/kaggle/input/tahapu/weak_100.txt")

    max_len_train = max(len(strong_train), len(weak_train))
    strong_train = (strong_train * ((max_len_train // len(strong_train)) + 1))[:max_len_train]
    weak_train = (weak_train * ((max_len_train // len(weak_train)) + 1))[:max_len_train]

    max_len_test = max(len(strong_test), len(weak_test))
    strong_test = (strong_test * ((max_len_test // len(strong_test)) + 1))[:max_len_test]
    weak_test = (weak_test * ((max_len_test // len(weak_test)) + 1))[:max_len_test]

    X_train, y_train = [], []
    X_test, y_test = [], []

    for seq in strong_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(1)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(1)
    for seq in weak_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(0)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(0)

    for seq in strong_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(1)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(1)
    for seq in weak_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(0)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(0)

    return X_train, y_train, X_test, y_test

def train_stage1():
    tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNA_bert_6", do_lower_case=False, trust_remote_code=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Prepare data
    X_train, y_train, X_test, y_test = prepare_stage1()
    print("Train Label Distribution:", Counter(y_train))
    print("Test Label Distribution:", Counter(y_test))

    train_ds = SimpleDataset(X_train, y_train, tokenizer)
    test_ds = SimpleDataset(X_test, y_test, tokenizer)
    batch_size = 8  # Starting batch size
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    model = DNABERT_BiLSTM(dropout_rate=0.3).to(device)  # Using DNA BERT + BiLSTM
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    # Learning rate scheduler (Cosine Annealing)
    scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

    best_f1 = 0
    for epoch in range(10):  # 10 epochs
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Stage 1 - Epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask).squeeze()
            loss = loss_fn(logits, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")
        evaluate(model, test_loader, device)

        # Update learning rate based on scheduler
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                label = batch["label"].to(device)
                logits = model(input_ids, attention_mask).squeeze()
                preds.extend((torch.sigmoid(logits) > 0.5).int().cpu().tolist())
                trues.extend(label.int().cpu().tolist())
        f1 = f1_score(trues, preds)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "enhancer_stage1.pth")
            print("✅ Stage 1 model saved.")

def train_stage2():
    tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNA_bert_6", do_lower_case=False, trust_remote_code=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Prepare data
    X_train, y_train, X_test, y_test = prepare_stage2()
    print("Train Label Distribution:", Counter(y_train))
    print("Test Label Distribution:", Counter(y_test))

    train_ds = SimpleDataset(X_train, y_train, tokenizer)
    test_ds = SimpleDataset(X_test, y_test, tokenizer)
    batch_size = 8  # Starting batch size
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    model = DNABERT_BiLSTM(dropout_rate=0.3).to(device)  # Using DNA BERT + BiLSTM
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    # Learning rate scheduler (Cosine Annealing)
    scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

    best_f1 = 0
    for epoch in range(10):  # 10 epochs
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Stage 2 - Epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask).squeeze()
            loss = loss_fn(logits, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")
        evaluate(model, test_loader, device)

        # Update learning rate based on scheduler
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                label = batch["label"].to(device)
                logits = model(input_ids, attention_mask).squeeze()
                preds.extend((torch.sigmoid(logits) > 0.5).int().cpu().tolist())
                trues.extend(label.int().cpu().tolist())
        f1 = f1_score(trues, preds)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "enhancer_stage2.pth")
            print("✅ Stage 2 model saved.")

# === MAIN ===
if __name__ == "__main__":
    train_stage1()
    train_stage2()


tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

configuration_bert.py:   0%|          | 0.00/807 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/zhihan1996/DNA_bert_6:
- configuration_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


vocab.txt:   0%|          | 0.00/28.7k [00:00<?, ?B/s]

...

Epoch 8 Loss: 0.4144
Accuracy: 0.7210 | F1 Score: 0.7529 | Sensitivity (SN): 0.8500 | Specificity (SP): 0.5920 | MCC: 0.4575 | AUC: 0.7386
Confusion Matrix:
 [[425, 204], [75, 296]]
✅ Stage 2 model saved.


Stage 2 - Epoch 9: 100%|██████████| 928/928 [12:41<00:00,  1.22it/s]


Epoch 9 Loss: 0.3856
Accuracy: 0.6170 | F1 Score: 0.5391 | Sensitivity (SN): 0.4480 | Specificity (SP): 0.7860 | MCC: 0.2486 | AUC: 0.7387
Confusion Matrix:
 [[224, 107], [276, 393]]


Stage 2 - Epoch 10: 100%|██████████| 928/928 [12:40<00:00,  1.22it/s]


Epoch 10 Loss: 0.3713
Accuracy: 0.7250 | F1 Score: 0.7581 | Sensitivity (SN): 0.8620 | Specificity (SP): 0.5880 | MCC: 0.4679 | AUC: 0.7425
Confusion Matrix:
 [[431, 206], [69, 294]]
✅ Stage 2 model saved.
